In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# 1. 自动定位项目根目录
project_root = Path.cwd().parent if "notebooks" in os.getcwd() else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.bsm_chooser import BsmChooserPricer

def run_stress_test():
    print("="*75)
    print(" [Week 7 Step 1] Chooser Option 定价模型极端压力测试 (Stress Testing)")
    print("="*75)
    
    # 2. 载入 Champion 模型
    model_path = project_root / "models" / "gbdt_residual_champion.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f" 找不到模型权重文件: {model_path}")
        
    models = joblib.load(model_path)
    pricer = BsmChooserPricer()
    
    # 3. 设定基准市场状态 (Baseline Regimes)
    base_s0 = 356.20        # 标的现价
    base_r = 0.0381         # 基准利率 3.81%
    base_q = 0.0168         # 滚动股息率 1.68%
    base_vol = 0.20         # 20D 实现波动率 20%
    base_vix = 0.18         # VIX 18.0
    
    # 构建基础特征字典 (与模型训练时的 12 维特征完全对齐)
    base_features = {
        'JPM_Close': base_s0,
        'VIX_Decimal': base_vix,
        'Risk_Free_Rate': base_r,
        'Daily_Return': 0.001,
        'Rolling_Vol_20d': base_vol,
        'Dividend_Growth_Proxy': base_q,
        'Real_Dividend_Yield': base_q,
        'VIX_JPM_Corr_20d': -0.35,
        'IR_Momentum_10d': 0.0001,
        'JPM_SMA20_Disparity': 0.015,
        'IV_RV_Spread': base_vix - base_vol,
        'Rate_Delta': 0.0005
    }

    # 4. 场景 1: 波动率暴涨/暴跌冲击测试 (Volatility Shocks: -30% ~ +100%)
    print("\n 场景 1: 波动率冲击测试 (Volatility Shock Analysis)")
    vol_shocks = [0.10, 0.15, 0.20, 0.30, 0.40, 0.50]  # 10% 到 50%
    
    vol_results = []
    for vol in vol_shocks:
        feat = base_features.copy()
        feat['Rolling_Vol_20d'] = vol
        feat['IV_RV_Spread'] = feat['VIX_Decimal'] - vol
        df_feat = pd.DataFrame([feat])
        
        row_res = {"Volatility": f"{vol*100:.0f}%"}
        for t1 in [0.25, 0.50, 0.75]:
            bsm_price = pricer.price_chooser(base_s0, 150.0, t1, 1.0, base_r, base_q, vol)
            res_pred = float(models[t1].predict(df_feat)[0])
            final_price = bsm_price + res_pred
            row_res[f"T1={t1} (BSM)"] = f"${bsm_price:.2f}"
            row_res[f"T1={t1} (Pred)"] = f"${final_price:.2f}"
            row_res[f"T1={t1} (Residual)"] = f"${res_pred:+.2f}"
        vol_results.append(row_res)
        
    df_vol_res = pd.DataFrame(vol_results)
    print(df_vol_res.to_string(index=False))

    # 5. 场景 2: 利率加息/降息冲击测试 (Interest Rate Shocks: -200bps ~ +300bps)
    print("\n 场景 2: 利率冲击测试 (Interest Rate Shock Analysis)")
    rate_shocks = [0.0181, 0.0281, 0.0381, 0.0481, 0.0581, 0.0681] # 1.81% 到 6.81%
    
    rate_results = []
    for r in rate_shocks:
        feat = base_features.copy()
        feat['Risk_Free_Rate'] = r
        df_feat = pd.DataFrame([feat])
        
        row_res = {"Rate": f"{r*100:.2f}%"}
        for t1 in [0.25, 0.50, 0.75]:
            bsm_price = pricer.price_chooser(base_s0, 150.0, t1, 1.0, r, base_q, base_vol)
            res_pred = float(models[t1].predict(df_feat)[0])
            final_price = bsm_price + res_pred
            row_res[f"T1={t1} (BSM)"] = f"${bsm_price:.2f}"
            row_res[f"T1={t1} (Pred)"] = f"${final_price:.2f}"
            row_res[f"T1={t1} (Residual)"] = f"${res_pred:+.2f}"
        rate_results.append(row_res)
        
    df_rate_res = pd.DataFrame(rate_results)
    print(df_rate_res.to_string(index=False))

    print("\n" + "="*75)
    print(" 压力测试完成！模型在波动率与利率极端变动下无数值崩塌，物理残差响应正常。")
    print("="*75)

if __name__ == "__main__":
    run_stress_test()

 [Week 7 Step 1] Chooser Option 定价模型极端压力测试 (Stress Testing)

 场景 1: 波动率冲击测试 (Volatility Shock Analysis)
Volatility T1=0.25 (BSM) T1=0.25 (Pred) T1=0.25 (Residual) T1=0.5 (BSM) T1=0.5 (Pred) T1=0.5 (Residual) T1=0.75 (BSM) T1=0.75 (Pred) T1=0.75 (Residual)
       10%       $205.87        $217.93            $+12.05      $205.87       $217.05           $+11.17       $205.87        $220.19            $+14.32
       15%       $205.87        $217.49            $+11.62      $205.87       $216.23           $+10.36       $205.87        $220.01            $+14.14
       20%       $205.87        $215.36             $+9.48      $205.87       $213.34            $+7.46       $205.87        $218.62            $+12.74
       30%       $205.90        $211.04             $+5.14      $205.90       $207.72            $+1.82       $205.91        $204.48             $-1.43
       40%       $206.29        $210.25             $+3.96      $206.30       $207.74            $+1.44       $206.42        $204.79    

In [11]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# 1. 自动定位项目根目录
project_root = Path.cwd().parent if "notebooks" in os.getcwd() else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.bsm_chooser import BsmChooserPricer

def run_stress_test():
    print("="*75)
    print("⚡ [Week 7 Step 1] Chooser Option 定价模型极端压力测试 (Stress Testing)")
    print("="*75)
    
    # 2. 载入 Champion 模型
    model_path = project_root / "models" / "gbdt_residual_champion.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"❌ 找不到模型权重文件: {model_path}")
        
    models = joblib.load(model_path)
    pricer = BsmChooserPricer()
    
    # 3. 设定基准市场状态 (Baseline Regimes)
    base_s0 = 356.20        # 标的现价
    base_r = 0.0381         # 基准利率 3.81%
    base_q = 0.0168         # 滚动股息率 1.68%
    base_vol = 0.20         # 20D 实现波动率 20%
    base_vix = 0.18         # VIX 18.0
    
    base_features = {
        'JPM_Close': base_s0,
        'VIX_Decimal': base_vix,
        'Risk_Free_Rate': base_r,
        'Daily_Return': 0.001,
        'Rolling_Vol_20d': base_vol,
        'Dividend_Growth_Proxy': base_q,
        'Real_Dividend_Yield': base_q,
        'VIX_JPM_Corr_20d': -0.35,
        'IR_Momentum_10d': base_r,   # 修正：基准动能设为当前基准利率
        'JPM_SMA20_Disparity': 0.015,
        'IV_RV_Spread': base_vix - base_vol,
        'Rate_Delta': 0.0005
    }

    # 4. 场景 1: 波动率暴涨/暴跌冲击测试 (联动更新 VIX 与 IV_RV_Spread)
    print("\n📊 场景 1: 波动率冲击测试 (Volatility Shock Analysis)")
    vol_shocks = [0.10, 0.15, 0.20, 0.30, 0.40, 0.50]  # 10% 到 50%
    
    vol_results = []
    for vol in vol_shocks:
        feat = base_features.copy()
        feat['Rolling_Vol_20d'] = vol
        
        # 🔑 修正 1：VIX 随实现波动率同步联动冲击 (保持合理的波动率溢价关系)
        vix_shocked = base_vix * (vol / base_vol)
        feat['VIX_Decimal'] = vix_shocked
        feat['IV_RV_Spread'] = vix_shocked - vol  # 重新计算价差
        
        df_feat = pd.DataFrame([feat])
        
        row_res = {"Volatility": f"{vol*100:.0f}%"}
        for t1 in [0.25, 0.50, 0.75]:
            bsm_price = pricer.price_chooser(base_s0, base_s0, t1, 1.0, base_r, base_q, vol) # 此处已调整为平值 K=S0
            res_pred = float(models[t1].predict(df_feat)[0])
            final_price = bsm_price + res_pred
            row_res[f"T1={t1} (BSM)"] = f"${bsm_price:.2f}"
            row_res[f"T1={t1} (Pred)"] = f"${final_price:.2f}"
            row_res[f"T1={t1} (Residual)"] = f"${res_pred:+.2f}"
        vol_results.append(row_res)
        
    df_vol_res = pd.DataFrame(vol_results)
    print(df_vol_res.to_string(index=False))

    # 5. 场景 2: 利率加息/降息冲击测试 (联动更新 IR_Momentum_10d 与 Rate_Delta)
    print("\n📊 场景 2: 利率冲击测试 (Interest Rate Shock Analysis)")
    rate_shocks = [0.0181, 0.0281, 0.0381, 0.0481, 0.0581, 0.0681] # 1.81% 到 6.81%
    
    rate_results = []
    for r in rate_shocks:
        feat = base_features.copy()
        feat['Risk_Free_Rate'] = r
        
        # 🔑 修正 2：联动更新利率衍生特征
        feat['IR_Momentum_10d'] = r          # 利率基准中枢随加息/降息同步调整
        feat['Rate_Delta'] = r - base_r      # 利率差分反映本次冲击的变动幅度
        
        df_feat = pd.DataFrame([feat])
        
        row_res = {"Rate": f"{r*100:.2f}%"}
        for t1 in [0.25, 0.50, 0.75]:
            bsm_price = pricer.price_chooser(base_s0, base_s0, t1, 1.0, r, base_q, base_vol)
            res_pred = float(models[t1].predict(df_feat)[0])
            final_price = bsm_price + res_pred
            row_res[f"T1={t1} (BSM)"] = f"${bsm_price:.2f}"
            row_res[f"T1={t1} (Pred)"] = f"${final_price:.2f}"
            row_res[f"T1={t1} (Residual)"] = f"${res_pred:+.2f}"
        rate_results.append(row_res)
        
    df_rate_res = pd.DataFrame(rate_results)
    print(df_rate_res.to_string(index=False))

    print("\n" + "="*75)
    print("✅ 压力测试完成！特征联动自洽，模型在极端变动下物理响应正常。")
    print("="*75)

if __name__ == "__main__":
    run_stress_test()

⚡ [Week 7 Step 1] Chooser Option 定价模型极端压力测试 (Stress Testing)

📊 场景 1: 波动率冲击测试 (Volatility Shock Analysis)
Volatility T1=0.25 (BSM) T1=0.25 (Pred) T1=0.25 (Residual) T1=0.5 (BSM) T1=0.5 (Pred) T1=0.5 (Residual) T1=0.75 (BSM) T1=0.75 (Pred) T1=0.75 (Residual)
       10%        $21.66         $33.13            $+11.47       $24.35        $32.36            $+8.02        $26.46         $41.14            $+14.67
       15%        $31.71         $42.19            $+10.48       $35.88        $43.68            $+7.80        $39.12         $53.01            $+13.89
       20%        $41.90         $51.20             $+9.30       $47.52        $53.65            $+6.13        $51.86         $64.15            $+12.29
       30%        $62.36         $65.64             $+3.29       $70.85        $70.61            $-0.24        $77.37         $79.16             $+1.79
       40%        $82.78         $85.26             $+2.48       $94.10        $92.93            $-1.17       $102.76        $100.72  